# 03 · Row check, gate and manifest

**Run first:**
```bash
uv run pytest -v tests/unit/dtrack_left/checks/test_row_check.py tests/unit/dtrack_right/checks/test_row_check.py \
  tests/unit/common/test_manifest.py tests/unit/test_protocol_flow.py
just e2e && jq '.summary, .gate_decision' examples/e2e/output/s3_mirror/row_check/orders.json
```
**Read:** `src/dtrack_right/checks/row_check.py`: `reconcile_partitions`, `evaluate_gate`,
`reconcile_pair`, `run_row_check`; and `common/manifest.py::build_manifest`.

In [ ]:
import nbkit
import pandas as pd
from nbkit import code, show, table

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## 1 · Four verdicts per partition

In [ ]:
from dtrack_right.checks.row_check import (
    evaluate_gate,
    reconcile_partitions,
    row_check_csv_bytes,
    summarize_partitions,
)

left = {"2026-01-14": 30, "2026-01-15": 30, "2026-01-31": 30}
right = {"2026-01-14": 30, "2026-01-15": 27, "2026-02-01": 5}
partitions = reconcile_partitions(left, right)
table(pd.DataFrame(partitions).T.rename_axis("partition"), index=True)
print("summary:", summarize_partitions(partitions))
code(
    row_check_csv_bytes({"pair": "orders", "partitions": partitions}).decode(),
    label="reports/orders_compare_row.csv",
)

## 2 · Three gate policies

`auto` passes when at least one partition matched; `ratio:x` also requires the excluded share
to be at most `x` (inclusive); `manual` never passes, it only reports.

In [ ]:
ten = {f"2026-01-{d:02d}": {"verdict": "match" if d <= 7 else "count_diff"} for d in range(1, 11)}
table(
    pd.DataFrame(
        [
            {
                "policy": policy,
                **{
                    k: evaluate_gate(ten, policy)[k]
                    for k in ("passed", "excluded_ratio", "matched_partitions", "total_partitions")
                },
            }
            for policy in ("auto", "ratio:0.3", "ratio:0.2", "manual")
        ]
    ).astype({"passed": str})
)

## 3 · Only matched partitions reach the manifest

In [ ]:
from dtrack_right.common.manifest import build_manifest

pair = {
    "left": {"source": "oracle", "table": "ORDERS", "date_col": "BUSINESS_DATE"},
    "right": {"source": "databricks", "table": "orders", "date_col": "business_date"},
    "col_map": {"AMOUNT": "amount", "STATUS": "status"},
    "vintage": "month",
}
document = {"partitions": partitions, "gate_decision": evaluate_gate(partitions, "auto")}
manifest = build_manifest("run", "orders", pair, document)
show(
    {k: manifest[k] for k in ("windows", "columns", "expected_units", "sample")},
    "manifest/orders.json (excerpt)",
)

## 4 · `manual` is terminal for a run

A pair with a verdict is no longer pending, and once every announced pair has a verdict the
row check closes. Re-running the same run with `gate=auto` changes nothing. To release a
reviewed pair, start a **new run** (new folder) with `auto` or `ratio:x`.

In [ ]:
from dtrack_right.checks.row_check import run_row_check as right_row_check

folder = nbkit.sandbox("03")
data, config, _ = nbkit.dataset(folder)
store = nbkit.VolumeStore(folder / "runs" / "manual")
nbkit.left_row_check(config, store, run_id="manual")
print(
    "manual:",
    right_row_check(store, nbkit.target_factory(data, folder), gate="manual"),
    "pair(s) reconciled",
)
print(
    "auto on the same run:",
    right_row_check(store, nbkit.target_factory(data, folder), gate="auto"),
    "pair(s) reconciled",
)
print("manifests:", store.list_keys("manifest/"), "| closed:", store.exists("row_check/_CLOSED"))

## 5 · The verdict is written last (a bug found by this course)

`reconcile_pair` used to write the verdict first. A crash before the manifest left the pair
"done" on restart, so a pair that **passed** the gate silently skipped Stage 2. Now the order is
CSV → manifest → verdict, the same rule as payload → pointer in notebook 02.

In [ ]:
store = nbkit.VolumeStore(folder / "runs" / "crash")
nbkit.left_row_check(config, store, run_id="crash")
real_put = store.put_json


def crash_on_manifest(key, value):
    if key.startswith("manifest/"):
        raise ConnectionError("process killed")
    return real_put(key, value)


store.put_json = crash_on_manifest
try:
    right_row_check(store, nbkit.target_factory(data, folder))
except ConnectionError as exc:
    print("crash:", exc)
store.put_json = real_put
print("verdicts after crash:", store.list_keys("row_check/"))
print("restart reconciles:", right_row_check(store, nbkit.target_factory(data, folder)), "pair(s)")
print("manifests:", store.list_keys("manifest/"))